# Ted Talk Topic Extraction with [NMF](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.pipeline import Pipeline

In [ ]:
# read data into pandas dataframes
transcript_df = pd.read_csv('data/transcripts.csv')
main_df = pd.read_csv('data/ted_main.csv', parse_dates=['film_date', 'published_date'],
                      converters={'ratings': literal_eval, 'tags': literal_eval, 'related_talks': literal_eval})

In [ ]:
#nltk.download('stopwords')
en_stop = set(nltk.corpus.stopwords.words('english'))

In [ ]:
# set model parameters
MIN_DF = 0.1
MAX_DF = 0.2
NUM_TOPICS = 10
MORE_STOPWORDS = set(['talk', 'applause', 'laughter', 'yeah', 'oh', 'guy'])

In [ ]:
# compile stopwords and create TFIDF vectorizer for transcript data
en_stop = en_stop | MORE_STOPWORDS
vectorizer = TfidfVectorizer(stop_words=en_stop,
                             use_idf=True,
                             ngram_range=(1, 1),
                             min_df=MIN_DF,
                             max_df=MAX_DF)

tfidf = vectorizer.fit_transform(transcript_df['transcript'])

# Fit NMF model

In [ ]:
# create, fit nmf model and transform tfidf vectorizer
nmf = NMF(n_components=NUM_TOPICS, random_state=42)
topics = nmf.fit_transform(tfidf)

# print top 5 words associated with each fitted topic
top_n_words = 5
t_words, word_strengths = {}, {}
for t_id, t in enumerate(nmf.components_):
    t_words[t_id] = [vectorizer.get_feature_names()[i] for i in t.argsort()[:-top_n_words - 1:-1]]
    word_strengths[t_id] = t[t.argsort()[:-top_n_words - 1:-1]]
for k, v in t_words.items():
    print(k,v)

# Testing out NMF fit on 5 randomly sampled talks

In [ ]:
# create sklearn pipeline for simple fit+transform of talks
pipe = Pipeline([
    ('tfidf', vectorizer),
    ('nmf', nmf)
])

# test out on 5 randomly sampled talks
rand_doc_ids = np.random.randint(0,len(transcript_df),5)
for doc_id in rand_doc_ids:
    t = pipe.transform([transcript_df['transcript'].iloc[doc_id]]) 
    print('Predicted most likely topic: ', t_words[np.argmax(t)])
    talk_data = main_df[main_df.url == transcript_df.iloc[doc_id]['url']]
    print('Talk Title:', talk_data.title.values)
    print('Actual Talk Tags:', talk_data.tags.values)
    print('----------------------------------------------------')


# applying topic extraction to every talk

In [ ]:
# define helper functions to retrieve topic and topic id for each talk in DF
def get_topic(transcript):
    t = pipe.transform([transcript])
    topic = t_words[np.argmax(t)]
    return topic

def get_topic_id(transcript):
    t = pipe.transform([transcript])
    return np.argmax(t)

In [ ]:
# get topic weights for each transcript
weights_df = pd.DataFrame(pipe.transform(transcript_df.transcript))
weights_df.columns = ['t1_weight',
                      't2_weight',
                      't3_weight',
                      't4_weight',
                      't5_weight',
                      't6_weight',
                      't7_weight',
                      't8_weight',
                      't9_weight',
                      't10_weight']

In [ ]:
# map helper functions to entire df
transcript_df['topic_pred'] = transcript_df.transcript.map(get_topic)
transcript_df['topic_pred_id'] = transcript_df.transcript.map(get_topic_id)

In [ ]:
transcript_df = transcript_df.merge(weights_df, left_index=True, right_index=True)

In [ ]:
# join on data from main_df and display top 5 rows
merged_df = main_df.merge(transcript_df, on='url')
merged_df.drop('transcript', axis=1, inplace=True)
merged_df.head(5)

# Data cleaning

In [ ]:
# get year for each talk
merged_df['film_year'] = pd.to_datetime(merged_df.film_date, unit='s').dt.year

In [ ]:
# get the top (most-selected) rating for each talk
def get_top_rating(rating):
    counts = [r['count'] for r in rating]
    return rating[np.argmax(counts)]['name']
merged_df['top_rating'] = merged_df.ratings.map(get_top_rating)

In [ ]:
# show counts of each top rating value
merged_df.groupby(['top_rating'])[['url']].count()

In [ ]:
# show counts of each predicted topic
merged_df.groupby(['topic_pred_id'])[['url']].count()

In [ ]:
# each row is a topic, each column is an associated word
pd.DataFrame(t_words).T

In [ ]:
# IGNORE
"""
# Emulate this with d3 for lollipop chart
topic_dict = {}
for i in range(len(t_words)):
    tag_dict = {}
    df_filtered = merged_df.loc[merged_df.topic_pred_id == i]
    for tag_list in list(df_filtered.tags):
        for tag in tag_list:
            if tag in tag_dict:
                tag_dict[tag] += 1
            else:
                tag_dict[tag] = 1
    top_ten_tags = heapq.nlargest(10, tag_dict.items(), key=operator.itemgetter(1))
    topic_dict[i] = top_ten_tags
topic_dict

# create demo topic/tag data for lollipop chart prototype, then write out to csv
demo_topic_tag_df = pd.DataFrame(topic_dict[0])
demo_topic_tag_df.columns = ['tag', 'count']
demo_topic_tag_df.to_csv('data/demo_topic_tag.csv')
"""

In [ ]:
merged_df.to_csv('data/labelled_data.csv')